In [1]:
#!pip install tensorflow

In [2]:
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
#from tensorflow.keras.datasets import mnist
#from tensorflow.keras.utils import to_categorical

In [3]:
#import tensorflow as tf
from tensorflow import keras

In [4]:
# Wczytanie i podział datasetu
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"Zbiór treningowy: {X_train.shape[0]}")
print(f"Zbiór testowy: {X_test.shape[0]}")

Zbiór treningowy: 60000
Zbiór testowy: 10000


In [5]:
# Reshaping i normalizacja
X_train = X_train.reshape(-1, 28, 28, 1) / 255.0
X_test = X_test.reshape(-1, 28, 28, 1) / 255.0

# One-hot encoding
y_train = keras.utils.to_categorical(y_train)
y_test = keras.utils.to_categorical(y_test)

In [6]:
# Budowa modelu CNN, jego konfiguracja, trening, ocena i zapis do pliku.
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(32, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(X_train, y_train, epochs=5, batch_size=64)

loss, acc = model.evaluate(X_test, y_test)
print(f"Accuracy: {acc:.2%}")

model.save('mnist_cnn.keras')

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.9536 - loss: 0.1580
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9858 - loss: 0.0461
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9904 - loss: 0.0322
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9923 - loss: 0.0237
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9941 - loss: 0.0174
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9908 - loss: 0.0318
Accuracy: 99.08%


In [7]:
# Funkcja przygotowująca własne zdjęcie cyfry
import numpy as np
from PIL import Image, ImageOps

def prepare_image_for_mnist(img_path):
    img = Image.open(img_path).convert('L')     # skala szarości
    img = ImageOps.autocontrast(img)            # poprawa kontrastu
    img = img.resize((28, 28))                  # dopasowanie rozmiaru

    img = np.array(img)

    # Jeśli własny obraz ma ciemną cyfrę na jasnym tle, odwracamy kolory
    img = 255 - img

    img = img / 255.0                           # normalizacja
    img = img.reshape(1, 28, 28, 1)             # format CNN
    
    return img

In [8]:
# Funkcja rozpoznawania cyfry z obrazu
#from tensorflow.keras.models import load_model
model = keras.models.load_model('mnist_cnn.keras')

def classify_image(img_path):
    img = prepare_image_for_mnist(img_path)
    predictions = model.predict(img, verbose=0)[0]
    digit = np.argmax(predictions)
    confidence = predictions[digit]

    #print(f"Plik: {img_path}")
    print(f"Rozpoznana cyfra: {digit}")
    print(f"Pewność modelu: {confidence:.2%}")

    #print(f"{img_path}: Digit {digit}")

In [9]:
# Wywołanie funkcji do rozponanwania cyfry
classify_image('img_digit.jpg')

Rozpoznana cyfra: 1
Pewność modelu: 91.99%
